In [ ]:
import sagemaker
import boto3
from botocore.exceptions import ClientError
import ast
import datasets
import time

AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
AWS_SESSION_TOKEN=

session = sagemaker.Session()

sagemaker_session_bucket=None
if sagemaker_session_bucket is None and session is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = session.default_bucket()
    
role = sagemaker.get_execution_role()

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
def get_secret_hf(session):

    secret_name = "hf-access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return ast.literal_eval(secret)['hf-access-token']

session = boto3.session.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

hf_token = get_secret_hf(session)

## Load in and Prep Data

In [ ]:
import pandas as pd
curated = pd.read_csv('data/curated_examples.csv', encoding='MacRoman')
curated = curated.dropna(axis=1, how='all')

prefix_PN = "This is a Progress Note - OT from "
prefix_SN = "This is a Scratch Note - OT from "
curated['PN'] = curated.apply(lambda row: prefix_PN + row['Date'] + ":\n" + row['PN'], axis=1)
curated['SN'] = curated.apply(lambda row: prefix_SN + row['Date'] + ":\n" + row['SN'], axis=1)

In [ ]:
curated_dict_list = curated.to_dict(orient="records")

In [ ]:
import datasets
dataset = datasets.Dataset.load_from_disk('data/ft_dataset_redact')

In [ ]:
sn_token_nums = []
pn_token_nums =[]

for x in range(0, len(curated)):
    input_ids_sn = llama_tokenizer(curated['SN'][x])["input_ids"]
    input_ids_pn = llama_tokenizer(curated['PN'][x])["input_ids"]
    num_input_tokens_sn = len(input_ids_sn)
    num_input_tokens_pn = len(input_ids_pn)
    sn_token_nums.append(num_input_tokens_sn)
    pn_token_nums.append(num_input_tokens_pn)

curated['sn_token_nums'] = sn_token_nums
curated['pn_token_nums'] = pn_token_nums

In [ ]:
curated = curated[curated['pn_token_nums']<800]

In [ ]:
curated = curated.reset_index()

In [ ]:
curated

In [ ]:
import matplotlib.pyplot as plt
plt.hist(dataset['token_nums'])

## Prepare Prompts

In [ ]:
generation_system_prompt = f"""<<SYS>>
You are a helpful, respectful, and honest assistant in a pediatric rehabilitation clinic. Your job is to help clinicians write concise, accurate documentation. The documentation should strictly be in point-form writing. Do not write in paragraphs. Write as concisely as possible while still being clear and accurate. Clearly distinguish between subjective reporting, observations, and treatment goals. Always answer as helpfully as possible, based on what you have been told, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature."""

generation_prompt_6 = f"""<<SYS>>\nYou are a helpful, respectful, and honest assistant in a pediatric rehabilitation clinic. You follow these rules:
1. Follow directions meticulously.
2. Write in point-form. Do not write in paragraphs.
3. Do not use full sentence structure. Write as concisely as possible, while being accurate.
4.  Do not produce any extra text. Only write what is asked for.
5. Clearly distinguish between reporting, observations, and goals.
6. Answer as helpfully as possible, while being safe.
"""

evaluation_2 = f"""<<SYS>>
You are a helpful, respectful, and honest assistant. Your task is to rank multiple Generated Scratch Notes by considering a set of criteria and an original Progress Note that all of the Generated Scratch Notes were based on. Follow these rules:
1. Follow directions meticulously.
2. All output must be in valid JSON. Don’t add explanation beyond the JSON."""


In [ ]:
def build_llama_prompt(content, system_prompt = 'default', id_string='00000'):
    start_prompt = "<s>[INST] "
    end_prompt = " [/INST]"
    if system_prompt == 'default':
        system_prompt = f"<<SYS>>\nYou are a helpful, respectful, and honest assistant in a pediatric rehabilitation clinic. Your job is to help clinicians write concise, accurate documentation. The documentation should strictly be in point-form writing. Do not write in paragraphs. Write as concisely as possible while still being clear and accurate. Clearly distinguish between subjective reporting, observations, and treatment goals. Always answer as helpfully as possible, based on what you have been told, while being safe. Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. Please ensure that your responses are socially unbiased and positive in nature."
        
    end_id = f"""\nThe ID for this generation is {id_string}.
<</SYS>>
"""
    
    return start_prompt + system_prompt + end_id + content + end_prompt

In [ ]:
def generation_prompt_template(pn, pn_id, demo1, demo2, output_num = 0):
    
    message_examples = ""

    message_examples = message_examples + f"""Here is a Progress Note in SOAP format. Write a matching Scratch Note.
{demo1['PN']} [/INST]
{demo1['SN']}
[INST] Here is a Progress Note in SOAP format. Please write a matching Scratch Note.
{demo2['PN']} [/INST]
{demo2['SN']}</s>"""


    message = message_examples + f"""<s>[INST] Here is a Progress Note in SOAP format. Write a matching Scratch Note.
{pn}"""

    id_string = f"{pn_id}, {demo1['ID']}, {demo2['ID']}, {output_num}"
    print(id_string)
    return message, id_string

    
    

In [ ]:
def create_generation_list_final(pn_list, curated_dict_list, sys_prompt, repeat = 1):
    
    prompts = []
    
    
    i = 0
    for x in range(len(pn_list)):
        for j in range(0, repeat):
            
            if i < len(curated_dict_list)-1:
                pair1 = curated_dict_list[i]
                if i + 1 < len(curated_dict_list):
                    pair2 = curated_dict_list[i + 1]
            
            else: 
                i = 0
                pair1 = curated_dict_list[i]
                if i + 1 < len(curated_dict_list):
                    pair2 = curated_dict_list[i + 1]
            i += 2
            
            if x==44 or x==1898:
                pair1 = curated_dict_list[0]
                pair2 = curated_dict_list[1]

            message, id_str = generation_prompt_template(pn_list[x], x, pair1, pair2, j)
            prompt = build_llama_prompt(content = message, system_prompt = sys_prompt, id_string= id_str)
            prompts.append(prompt)
        

    return prompts


In [ ]:
progress_notes_list = dataset['input_text']

# First demo pair generation
# prompt_list = create_generation_list(progress_notes_list, demos_dict_list[:2], generation_prompt_6, repeat = 5)

# Second demo pair generation

prompt_list = create_generation_list_final(progress_notes_list, curated_dict_list, generation_prompt_6)


In [ ]:
from transformers import LlamaTokenizer
llama_tokenizer = LlamaTokenizer.from_pretrained('meta-llama/Llama-2-70b', token=hf_token)

def check_prompt_tokens(list_of_prompts, threshold):
    num_token_list = []
    for i in range(len(list_of_prompts)):  
        input_ids = llama_tokenizer(list_of_prompts[i])["input_ids"]
        num_input_tokens = len(input_ids)
        num_token_list.append(num_input_tokens)
        if num_input_tokens > threshold:
            print("Number of tokens in the prompt: " + str(num_input_tokens) + " ID: " + str(i) )
    
    result = all(num < threshold for num in num_token_list)
    return result

In [ ]:
check_prompt_tokens(prompt_list, 3500)

In [ ]:
prompt_list

## Run Model

In [ ]:
from sagemaker.huggingface import get_huggingface_llm_image_uri

# retrieve the llm image uri
llm_image = get_huggingface_llm_image_uri(
  "huggingface",
  version="1.0.3"
)

# print ecr image uri
print(f"llm image uri: {llm_image}")


In [ ]:
import json
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.async_inference.async_inference_config import AsyncInferenceConfig

# sagemaker config
instance_type = "ml.g5.48xlarge"
number_of_gpu = 8
health_check_timeout = 600

# TGI config
config = {
  'HF_MODEL_ID': "TheBloke/Llama-2-70B-Chat-GPTQ", # model_id from hf.co/models
  'SM_NUM_GPUS': json.dumps(number_of_gpu), # Number of GPU used per replica
  'MAX_INPUT_LENGTH': json.dumps(3500),  # Max length of input text
  'MAX_TOTAL_TOKENS': json.dumps(4096),  # Max length of the generation (including input text)
  'MAX_BATCH_TOTAL_TOKENS': json.dumps(8192),  # Limits the number of tokens that can be processed in parallel during the generation
  'HF_MODEL_QUANTIZE': "gptq", 
  'HUGGING_FACE_HUB_TOKEN': hf_token
}

async_config = AsyncInferenceConfig(
    output_path= "s3://eko-ekoka-ai-project/data/generation_output/Feb29_generation_1" ,
)


# create HuggingFaceModel
llm_model = HuggingFaceModel(
  role=role,
  image_uri=llm_image,
  env=config
)

async_predictor = llm_model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    async_inference_config=async_config,
    tags=[{"Key":'uw-ai-note-generation', "Value":'first-stage-generation'}],
    vpc_config_override = {"Subnets": ['subnet-0ba981d4c6314f9a9'], "SecurityGroupIds": ['sg-04645476ef00795e3'] },
    
)




In [ ]:
def run_model(list_of_prompts):

    response_list = []
    start = time.time()

    for prompt in list_of_prompts:
        input_ids = llama_tokenizer(prompt)["input_ids"]
        num_input_tokens = len(input_ids)
        max_tokens = 4096 - num_input_tokens

        payload = {
      "inputs": prompt,
      "parameters": {
        "do_sample": True,
        "top_p": 0.9,
        "temperature": 0.8,
        "max_new_tokens": max_tokens,
        "repetition_penalty": 1.03,
        "stop": ["\nUser:","<|endoftext|>","</s>"]
        }
        }


        response = async_predictor.predict(payload)
        response_list.append(response)

    print(f"Time taken: {time.time() - start}s")
    return response_list

In [ ]:
generation_response_list = run_model(prompt_list)

## Ranking Evaluation

In [ ]:
def extract_generated_output(text):
    text = str(text)
    split_text = text.split('[/INST]')
    if len(split_text) > 1:
        return split_text[-1].strip()
    else:
        return ""
    
def extract_inference_note(text):
    text = str(text)
    split_text = text.split('[INST]')
    text = split_text[-1]
    PN_split = text.split('[/INST]')
    PN = PN_split[0]
    PN = PN.replace("Here is a Progress Note in SOAP format. Write a matching Scratch Note.", " ")
    PN = PN.replace('Here is a more concise version of the Scratch Note.', '')
    
    
    return(PN)

def create_generation_output_df(list_of_responses):
    df = pd.DataFrame({'output': [d[0]['generated_text'] for d in list_of_responses]})
    
    pattern = r'(\d+), (\d+), (\d+), (\d+)'

    df['generated_output'] = df['output'].apply(extract_generated_output)
    df['historical_note'] = df['output'].apply(extract_inference_note)

    df[['Note_id', 'demo_id1', 'demo_id2', 'output_num']] = df['output'].str.extract(pattern)
    df['generated_output'] = df['generated_output'].str.replace(r'"}]', '')
    
    return df
    

In [ ]:
output_df = create_generation_output_df(generation_response_list)

In [ ]:
output_df


In [ ]:
def ranking_prompt_template(pn, pn_id, sn_list, example_sn):
    content = f"""Please rank the Generated Scratch Notes based on the all of the following criteria: 
Criterion 1: Distinguishes between observations, subjective reporting, and goals.
Criterion 2: Accuracy: how faithfully details from the original Progress Note are captured.
Criterion 3: Realism: how much it looks like a real Scratch Note.
For context, here is an example of a real Scratch Note: {example_sn}
All the Generated Scratch Notes are based on the following original Progress Note: {pn} Compare the Generated Scratch Notes and their details to this original Progress Note when ranking the Generated Scratch Notes.
    """
    
    i = 0
    for sn in sn_list:
        i+=1
        content += f"""Generated Scratch Note {i}: {sn['generated_output']}
        """
        example_sn = sn
        
        
    content += f"""Output the ranked order of the Generated Scratch Notes based on the provided Criteria. As a final answer, provide the ranked order in valid JSON format like the following {{"1": Generated Scratch Note X, "2": Generated Scratch Note Y}}."""
    id_string = f"{pn_id}, {example_sn['demo_id1']}, {example_sn['demo_id2']}"
    
    return content, id_string

In [ ]:
def create_ranking_list(df, example_sn, sys_prompt):

    grouped = df.groupby(['Note_id', 'demo_id1', 'demo_id2'])

    sn_compare_list = []
    for name, group in grouped:
        # print("Group:", name)
        # print(group['generated_output'])

        sn_dict = group.to_dict(orient='records')

        sn_compare_list.append(sn_dict)
       
    ranking_prompt_list = []
    for compare_sn in sn_compare_list:
        sn_sample = compare_sn[0]
        message, id_str = ranking_prompt_template(sn_sample['historical_note'], sn_sample['Note_id'], compare_sn, example_sn)
        prompt = build_llama_prompt(content = message, system_prompt = sys_prompt, id_string= id_str)
        ranking_prompt_list.append(prompt)
    
    return ranking_prompt_list
    
    

In [ ]:
prompt_rank = create_ranking_list(output_df_2, sn_real_6, evaluation_1)


In [ ]:
check_prompt_tokens(prompt_rank, 3750)

In [ ]:
prompt_rank

In [ ]:
rank_response_list_1 = run_model(prompt_rank)
# rank_response_list_2 = run_model(prompt_rank)

In [ ]:
rank_response_list_1

In [ ]:
llm.delete_model()
llm.delete_endpoint()